In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_lens import HookedTransformer
import time
import os

In [ ]:

model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_hf = AutoModelForCausalLM.from_pretrained(model_name)
model_hf = model_hf.to("cuda")
model_tl = HookedTransformer.from_pretrained_no_processing(model_name, device="cuda")

In [ ]:
conv = [
    {"role": "user", "content": "Exolain in detail how the a combustion engine works."},
]
convs = [conv] * 32
inputs = tokenizer.apply_chat_template(
    convs,
    return_dict=True,
    return_tensors="pt",
    tokenize=True
)
inputs = {k: v.to("cuda") for k, v in inputs.items()}
print(inputs["input_ids"].shape)

In [ ]:
# Calculate duration
start_time = time.time()
response = model_hf.generate(
    **inputs,
    max_new_tokens=1000,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)
end_time = time.time()
duration_hf = end_time - start_time
print("HF duration:", duration_hf)


In [ ]:
start_time = time.time()
response = model_tl.generate(
    inputs["input_ids"],
    max_new_tokens=1000,
    do_sample=False,
    verbose=False,
)
end_time = time.time()
duration_tl = end_time - start_time
print("TL duration:", duration_tl)

In [ ]:
print("HF impovement over TL:", (1 - duration_hf/duration_tl) * 100)

## Compare trasnformer lens cache to return hidden states of huggging face

In [ ]:
prompt = "Explain in detail how the a combustion engine works."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
print(inputs["input_ids"].shape)

Huggingface

We need to set hugging face model final layer norm to identity to have the last layer output return in output hidden states

In [ ]:
original_norm = model_hf.model.norm
class IdentityNorm(torch.nn.Module):
    def forward(self, x):
        return x
identity_norm = IdentityNorm()
model_hf.model.norm = identity_norm
print(model_hf)
print(original_norm)

Hidden state tuple include the output of the embedding layer also!

In [ ]:
model_hf.model.norm = identity_norm
with torch.no_grad():
    output = model_hf(**inputs, output_hidden_states=True)
hidden_states_hf = output.hidden_states[1:]
print(len(hidden_states_hf))

TransformerLens


In [ ]:
hookpoints = [f"blocks.{i}.hook_resid_post" for i in range(model_tl.cfg.n_layers)]
with torch.no_grad():
    logits, output_tl = model_tl.run_with_cache(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        names_filter=hookpoints,
    )
print(output_tl)


In [ ]:
for i ,h_hf, h_tl in zip(range(len(hidden_states_hf)), hidden_states_hf, output_tl.values()):
    # if i == 27:
    #     h_tl = model_tl.ln_final(h_tl)
    max_diff = (h_hf - h_tl).abs().max().item()
    mean_diff = (h_hf - h_tl).abs().mean().item()
    print(torch.allclose(h_hf, h_tl, atol=1e-1), f"Max diff: {max_diff}, Mean diff: {mean_diff}")

In [ ]:
last_hidden_hf = hidden_states_hf[-1]
last_hidden_tl = list(output_tl.values())[-1]
print(last_hidden_hf.shape, last_hidden_tl.shape)

In [ ]:
print("HF:")
print(" mean", last_hidden_hf.mean().item())
print(" std", last_hidden_hf.std().item())
print("TL")
print(" mean", last_hidden_tl.mean().item())
print(" std", last_hidden_tl.std().item())

In [ ]:
prompts = [
    [
        {"role": "user", "content": "Explain in detail how the a combustion engine works."}
    ],
    [
        {"role": "user", "content": "Say only the word hello."}
    ]
]
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(
    prompts,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
    padding=True,
    tokenize=True
)
inputs = {k: v.to("cuda") for k, v in inputs.items()}
for prompt in tokenizer.batch_decode(inputs["input_ids"]):
    print("\n")
    print( prompt)

In [ ]:
responses = model_hf.generate(
    **inputs,
    max_new_tokens=5,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)
for ans in tokenizer.batch_decode(responses):
    print('\n')
    print(ans)

In [ ]:
query_length = inputs["input_ids"].shape[1]
for res in tokenizer.batch_decode(responses[:, query_length:]):
    print("\n")
    print( res)

In [ ]:
model_hf.config

In [ ]:
llama = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", device_map="auto")

In [ ]:
llama.device

In [ ]:
!nvidia-smi